# GreenGate

Three stories, each the same four steps:

**load the two models  ->  audit  ->  use it  ->  summary**

| part | models |
|---|---|
| 1 | text - Qwen-1.5B and Mistral-7B |
| 2 | vision - SmolVLM and Qwen2-VL |
| 3 | API - two hosted models |

Run everything once before the viva. On the day, only the question cells in Part 1 change.

---

In [ ]:
!pip install -q "greengate[gpu,api]"

import greengate, torch, textwrap, time
print('greengate', greengate.__version__)
print('GPU      ', torch.cuda.get_device_name(0)
      if torch.cuda.is_available() else 'none - carbon would be estimated')

### A sample of the developer's own traffic

`audit()` needs a few dozen real queries to measure the two tiers on. These stand in for whatever the application actually receives.

In [ ]:
QUERIES = [
    'What is the capital of Japan?',
    'What does HTTP stand for?',
    'Who wrote Hamlet?',
    'What is 15 percent of 200?',
    'What does CPU stand for?',
    'How many continents are there?',
    'What is the chemical symbol for gold?',
    'What is the capital of Canada?',
    'Who painted the Mona Lisa?',
    'What is the largest ocean on Earth?',
    'What gas do plants absorb?',
    'What does SQL stand for?',
    'Who discovered penicillin?',
    'What does DNS stand for?',
    'What is the chemical formula for water?',
    'What does GPU stand for?',
    'What is 7 times 12?',
    'What does API stand for?',
    'What is the capital of Egypt?',
    'How many hours are in a week?',
    'Explain what a database index does.',
    'Why is the sky blue?',
    'Explain recursion to a beginner.',
    'What is the difference between TCP and UDP?',
    'Explain what caching is and why it helps.',
    'What is a hash function used for?',
    'Explain the difference between a compiler and an interpreter.',
    'What is the difference between a process and a thread?',
    'Explain what a REST API is.',
    'What is a memory leak?',
    'Explain the difference between latency and bandwidth.',
    'What is the purpose of a load balancer?',
    'Explain what containerisation provides.',
    'What problem does a message queue solve?',
    'Explain what unit testing is for.',
    'Should a startup choose SQL or NoSQL, and why?',
    'Is remote work better than office work?',
    'When is eventual consistency unacceptable in a payment system?',
    'What are the trade-offs of microservices for a team of five?',
    'How would you explain technical debt to a non-technical manager?',
    'Is test coverage a good measure of software quality?',
    'How should a team handle disagreement about architecture?',
    'What makes documentation actually useful?',
    'How should an engineer decide what to learn next?',
    'When should a company train its own model rather than use an API?',
    'How do you balance speed of delivery against reliability?',
    'Should every service be designed to scale from day one?',
    'What are the risks of relying on a single cloud provider?',
    'When is it right to rewrite rather than refactor?',
    'How should a team approach reducing its cloud carbon footprint?'
]

print(len(QUERIES), 'sample queries')

---
# Part 1 - text

## Step 1: the developer points at the two models they already run

In [ ]:
gw = greengate.GreenGate(
    small='Qwen/Qwen2.5-1.5B-Instruct',
    large='mistralai/Mistral-7B-Instruct-v0.2',
    large_4bit=True,
    max_new_tokens=60,
)
print('ready')

That is the whole setup. Four arguments.

## Step 2: audit - should they cascade at all?

`audit()` runs **both** models over a sample of their own queries and measures what each tier really costs. From that comes the break-even point: the escalation rate at which cascading stops saving anything.

In [ ]:
report = gw.audit(QUERIES[:15], verbose=False)

print('cost ratio     ', round(report['cost_ratio_small_over_large'], 4))
print('break-even     ', f"{report['break_even_escalation_rate']:.1%}")
print()
print('recommendation:')
for line in textwrap.wrap(report['recommendation'], 66):
    print('  ', line)

## Step 3: set the operating point and start using it

Break-even is a cliff edge, not a target, so the gate is set to **half** of it. The threshold itself is a percentile of the developer's own traffic, so it needs to see some traffic first.

In [ ]:
be = report['break_even_escalation_rate']
for q in QUERIES:
    gw.route(q)                      # ordinary traffic, answered normally

gw.config(escalation_rate=be / 2)
print(f'operating at {be / 2:.0%} of traffic,'
      f' threshold {gw.profile()["threshold"]:.3f}')

## Step 4: answer questions

`route()` is the only call a developer makes from here on.

In [ ]:
def ask(question, gate=None):
    g = gate or gw
    r = g.route(question)
    tier = 'LARGE' if r.decision == 'ESCALATE' else 'small'
    print('Q:', question)
    print(f'  {tier} tier   doubt {r.signal:.2f} vs threshold {r.threshold:.2f}')
    print(f'  carbon {r.carbon_g:.6f} g', end='')
    print(f'   (incl. {r.wasted_carbon_g:.6f} g discarded)'
          if r.wasted_carbon_g else '')
    print()
    for line in textwrap.wrap(r.response.strip()[:400], 66):
        print('  ', line)
    return r

_ = ask('What is the capital of Australia?')

### <- Change the question here and run

In [ ]:
_ = ask('')

In [ ]:
_ = ask('')

In [ ]:
_ = ask('')

## Step 5: what did it cost?

`profile()` is what a developer checks in production.

In [ ]:
p = gw.profile()
print(f"queries           {p['queries']}")
print(f"escalated         {p['escalation_rate']:.1%}")
print(f"total carbon      {p['total_carbon_g']:.4f} g")
print(f"wasted on discards{p['wasted_carbon_g']:>9.4f} g")
print()
for line in textwrap.wrap(p.get('verdict', 'no verdict yet'), 66):
    print(' ', line)

## The extras

Five more things the library does: a named mode, a carbon ceiling, measuring arbitrary work, batched-service accounting, and fitting a calibration.

**1. Modes** - a named position on the escalation dial, for a developer who has not measured anything yet. `balanced` is the default.

In [ ]:
import greengate.core as _c

for m in ('green', 'balanced', 'quality'):
    gw.config(mode=m)
    rate = 1 - _c.AUTO_THRESHOLD_PERCENTILE[m] / 100
    print(f'  {m:9s} escalates {rate:4.0%}   threshold '
          + format(gw.profile()['threshold'], '.3f'))

gw.config(escalation_rate=be / 2)
print()
print(f'measured operating point {be / 2:.0%}   threshold '
      + format(gw.profile()['threshold'], '.3f'))

**2. A carbon ceiling** - when escalating would breach it the router returns the small answer instead of failing.

In [ ]:
gw.config(budget_g=0.00001)
for q in QUERIES[-2:]:
    print(' ', gw.route(q).decision)
print('  blocked:', gw.profile()['budget_blocked'], '- degraded, not failed')
gw.config(budget_g=1e9)

**3. `measure()`** - profile any block of GPU work, routed or not.

In [ ]:
from greengate import measure, ServiceLedger, should_cascade

with measure(n_queries=3, label='three answers') as m:
    for q in QUERIES[:3]:
        gw._small.generate(q)
print(m.report())

**4. `ServiceLedger`** - a production server batches requests, so the meter cannot say which watt belonged to which request. The ledger records a **window of work** with the tier that served it, and derives everything from the totals.

In [ ]:
led = ServiceLedger()

with measure(n_queries=8, label='small window') as w_small:
    for q in QUERIES[:8]:
        gw._small.generate(q)
led.add(w_small, tier='small')
print('window 1: 8 requests, small tier, '
      + format(w_small.carbon_grams, '.6f') + ' g')

with measure(n_queries=2, label='large window') as w_large:
    for q in QUERIES[-2:]:
        gw._large.generate(q)
led.add(w_large, tier='large')
print('window 2: 2 requests, large tier, '
      + format(w_large.carbon_grams, '.6f') + ' g')

rep = led.report()
LABEL = {
    'small_tier_queries':  'requests served by the small tier',
    'large_tier_queries':  'requests escalated to the large tier',
    'escalation_rate':     'escalation rate, derived from the counts',
    'total_energy_joules': 'energy across both windows (J)',
    'total_carbon_grams':  'carbon across both windows (g)',
    'wasted_carbon_grams': 'carbon spent on discarded small runs (g)',
    'contended_windows':   'windows that shared the GPU (0 = clean)',
    'cost_ratio_small_over_large': 'cost ratio, small over large',
    'break_even_escalation_rate':  'break-even escalation rate',
    'predicted_saving':    'saving at the realised rate',
}
print()
print('WHAT THE LEDGER DERIVES')
print('-' * 66)
for k, text in LABEL.items():
    if k in rep:
        v = rep[k]
        shown = format(v, '.6f') if isinstance(v, float) else str(v)
        print('  ' + shown.rjust(12) + '   ' + text)
print()
for line in textwrap.wrap('VERDICT: ' + rep['verdict'], 64):
    print(' ', line)
print()
print('Same arithmetic as profile(), but every per-request figure here is')
print('a total divided by a count, not a measurement. That is the honest')
print('option when requests are batched and cannot be separated.')

**5. `calibrate()`** - fits the overconfidence correction for a model with no shipped preset, on held-out MMLU questions where the right answer is known.

Two cautions, both handled below. It loads a **second copy** of the model, and it **overwrites the temperature in place** - which would shift every doubt score and invalidate the threshold already set. The original is restored afterwards.

In [ ]:
T_before = gw.temperature
print('current temperature (shipped preset): ' + format(T_before, '.4f'))
print()
try:
    fitted = gw.calibrate(n=30)       # 150 in the thesis; 30 to keep it quick
    print()
    print('  fitted T  ', fitted['temperature'])
    print('  ECE       ', fitted['ece_before'], '->', fitted['ece_after'])
except Exception as e:
    print('  skipped (' + type(e).__name__ + ') - needs the MMLU download',
          'and room for a second copy of the model')
finally:
    gw.temperature = T_before
    gw._small.T = T_before
    print()
    print('temperature restored to ' + format(gw.temperature, '.4f')
          + ' - the operating point set earlier is still valid')

### And what happens when the small tier is **not** the cheap one

Cascading only pays while the small tier is cheaper. To show what the framework does when it is not, the two tiers are swapped - Mistral is asked to play the small role - and audited exactly as before. `audit()` touches no gate state, so this is safe and reversible.

In [ ]:
gw._small, gw._large = gw._large, gw._small
gw.small_name, gw.large_name = gw.large_name, gw.small_name

bad = gw.audit(QUERIES[:8], verbose=False)

print('INVERTED PAIR   Mistral-7B  ->  Qwen-1.5B')
print('-' * 66)
print('  small tier costs ' + format(bad['mean_small_carbon_g'], '.6f') + ' g')
print('  large tier costs ' + format(bad['mean_large_carbon_g'], '.6f') + ' g')
print('  cost ratio       ' + format(bad['cost_ratio_small_over_large'], '.4f')
      + '   (above 1.0 means the small tier is the dearer one)')
print()
for line in textwrap.wrap('RECOMMENDATION: ' + bad['recommendation'], 64):
    print(' ', line)

gw._small, gw._large = gw._large, gw._small
gw.small_name, gw.large_name = gw.large_name, gw.small_name
print()
print('tiers restored:', gw.small_name.split('/')[-1], '->',
      gw.large_name.split('/')[-1])
print()
print('The framework returns a negative answer about itself rather than')
print('inventing a saving. The thesis reports a real case: a vision pair')
print('with a cost ratio of 1.049, rejected before any routing was tried.')

## Part 1 summary

In [ ]:
print('TEXT  Qwen-1.5B -> Mistral-7B')
print(f"  small tier costs {report['mean_small_carbon_g']:.6f} g per answer")
print(f"  large tier costs {report['mean_large_carbon_g']:.6f} g per answer")
print(f"  so cascading saves until {be:.0%} of queries escalate")
text_ratio = report['cost_ratio_small_over_large']

---
# Part 2 - vision

Same four steps. The signal here is **mean token probability** rather than entropy, because vision answers are only a word or two long - and it is the strongest signal measured anywhere in this study, AUROC 0.812 against 0.595 for text.

## Step 1: the two vision models

In [ ]:
from greengate.visiongen import SmallVisionModel

vsmall = SmallVisionModel('HuggingFaceTB/SmolVLM-256M-Instruct', max_new_tokens=40)
vlarge = SmallVisionModel('Qwen/Qwen2-VL-7B-Instruct', max_new_tokens=40,
                          load_in_4bit=True)
print('ready')

## Step 2: run both on the same image

In [ ]:
import io
from PIL import Image, ImageDraw

try:
    import requests
    URL = ('https://upload.wikimedia.org/wikipedia/commons/thumb/'
           '4/4d/Cat_November_2010-1a.jpg/320px-Cat_November_2010-1a.jpg')
    img = Image.open(io.BytesIO(requests.get(URL, timeout=10).content)).convert('RGB')
except Exception:
    img = Image.new('RGB', (320, 240), 'white')
    dr = ImageDraw.Draw(img)
    for x in (60, 150, 240):
        dr.ellipse([x - 30, 90, x + 30, 150], fill='steelblue')
display(img)

Q = 'What is in this picture?'
vs = vsmall.answer(img, Q)
vl = vlarge.answer(img, Q)
for name, v in (('small', vs), ('large', vl)):
    print(f'{name:6s} {v.response.strip()[:60]!r}')
    print(f'       confidence {v.avg_token_prob:.3f}'
          f'   carbon {v.carbon_grams:.6f} g')

## Step 3: the same verdict, from the same condition

In [ ]:
from greengate import should_cascade

vision_ratio = vs.carbon_grams / vl.carbon_grams
print(f'cost ratio {vision_ratio:.4f}')
print()
for rate in (0.0, 0.25, 0.5):
    print(f'  {rate:4.0%} ->', should_cascade(vs.carbon_grams,
                                              vl.carbon_grams, rate)['verdict'][:72])

`route()` takes text only in version 0.5.0, so vision uses the components directly. The signal works, the measurement works, the condition works - the image argument on `route()` is the next release, and it is in future work.

---
# Part 3 - two hosted models

A developer whose models both sit behind an API cannot route on confidence: the signal is entropy over the full vocabulary, and an API returns at most the top twenty tokens.

But the **accounting still works**, and that is the half that decides whether to cascade at all.

## Step 1: the two hosted models

In [ ]:
import os
from greengate.api_tier import APILargeTier

try:
    from kaggle_secrets import UserSecretsClient
    os.environ['OPENAI_API_KEY'] = UserSecretsClient().get_secret('OPENAI_API_KEY')
except Exception:
    pass
HAVE_KEY = bool(os.environ.get('OPENAI_API_KEY'))

# The wider the gap between these two, the clearer the cost ratio.
# Edit SMALL_API / LARGE_API to whatever your account can reach.
SMALL_API = 'gpt-4o-mini'
LARGE_API = 'gpt-4o'          # try a larger one here if you have access
FALLBACK  = 'gpt-4o'          # used if LARGE_API is not available

api_small = APILargeTier(SMALL_API, max_tokens=60, dry_run=not HAVE_KEY)
api_large = APILargeTier(LARGE_API, max_tokens=60, dry_run=not HAVE_KEY)

if HAVE_KEY:                   # confirm the large model actually answers
    try:
        api_large.query('ping')
    except Exception as e:
        print(LARGE_API, 'unavailable (' + type(e).__name__ + '),',
              'falling back to', FALLBACK)
        LARGE_API = FALLBACK
        api_large = APILargeTier(LARGE_API, max_tokens=60)

print('live' if HAVE_KEY else 'dry run - no key')
print(SMALL_API, '->', LARGE_API)

## Step 2: measure both, then ask the condition

In [ ]:
if HAVE_KEY:
    s = sum(api_small.query(q).carbon_grams for q in QUERIES[:5]) / 5
    l = sum(api_large.query(q).carbon_grams for q in QUERIES[:5]) / 5
else:
    s, l = 0.060, 1.000          # the figures reported in Chapter 5

print(SMALL_API, 'costs', format(s, '.6f'), 'g per answer')
print(LARGE_API, 'costs', format(l, '.6f'), 'g per answer')
print('cost ratio', format(s / l, '.4f'),
      '  break-even', format(1 - s / l, '.1%'))
print()
for rate in (0.0, 0.25, 0.5, 0.94):
    print(f'  {rate:4.0%} ->', should_cascade(s, l, rate)['verdict'][:72])
api_ratio = s / l

---
# All three, side by side

In [ ]:
print(f"{'':10s}{'ratio':>9s}{'break-even':>13s}")
print('-' * 32)
for name, r in (('text', text_ratio), ('vision', vision_ratio), ('api', api_ratio)):
    print(f'{name:10s}{r:9.4f}{1 - r:12.1%}')
print()
print('Three different break-even points, measured on the same afternoon.')
print('None of them could have been read off a specification sheet.')

---

**That is the argument.** You cannot know whether cascading helps you until you measure your own two models on your own hardware - and measuring it is what this library is for.

For a developer the whole thing is four lines:

```python
gw = greengate.GreenGate(small='...', large='...')
gw.audit(my_queries)      # should I cascade?
r = gw.route(query)       # answer one
gw.profile()              # what did it cost?
```